# Task 1: Ensemble learning on the HW4 data

**Goal:** predict the probability of Class 2 for each test row; performance is scored by AUC.

**Approach**
1. Check the class balance and preprocess (one-hot encode categoricals; scale numerics only for the linear model).
2. Tune four *different* base models (random forest, extra trees, gradient boosting, logistic regression) with randomized search and 5-fold CV on a training split.
3. Combine them into ensembles (soft voting, stacking) and compare against the single models using CV AUC.
4. Check the winner on a held-out validation split, then refit on all labeled data and write `test_result_HW6Ensemble.csv`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (ExtraTreesClassifier, RandomForestClassifier, HistGradientBoostingClassifier,
                              VotingClassifier, StackingClassifier)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import (train_test_split, RandomizedSearchCV, StratifiedKFold,
                                     cross_val_score, ParameterGrid)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 72

## 1. Load data and inspect the class balance

In [ ]:
train = pd.read_csv("train_dataHW4.csv")
test = pd.read_csv("test_dataHW4.csv")

X = train.drop(columns="Class")   # features
y = train["Class"]                # labels (we predict the probability of Class 2)

print("Train shape:", X.shape, "| Test shape:", test.shape)
print("Class balance:\n", y.value_counts(normalize=True).round(3))
print("Missing values in train:", int(X.isna().sum().sum()))

## 2. Preprocessing
Categorical columns are one-hot encoded (unseen categories in the test set are ignored rather than raising an error). Tree models don't need scaling, but logistic regression does, so there are two preprocessors.

In [ ]:
# object/category/string covers both old and new pandas string dtypes
cat_cols = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()
numeric_cols = X.columns.difference(cat_cols).tolist()

def make_preprocess(scale_numeric):
    num = StandardScaler() if scale_numeric else "passthrough"
    return ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
        ("num", num, numeric_cols)])

# Hold out 20% of labeled rows ONLY for the final honest check; all tuning uses X_train.
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

## 3. Tune each base model
Each model gets its own randomized search (AUC, 5-fold CV, training split only). Beyond depth and leaf size, I also search `max_features` (how many features each split sees), which controls how decorrelated the trees are, and the learning rate/leaf count for boosting.

In [ ]:
def tune(name, estimator, space, scale, n_iter):
    # Pipeline = preprocessing + model, so encoding is re-fit inside every CV fold (no leakage)
    pipe = Pipeline([("prep", make_preprocess(scale)), ("model", estimator)])
    n_iter = min(n_iter, len(ParameterGrid(space)))
    search = RandomizedSearchCV(pipe, space, n_iter=n_iter, scoring="roc_auc", cv=cv,
                                random_state=SEED, n_jobs=-1, return_train_score=True)
    search.fit(X_train, y_train)
    print(f"{name:>20}: best CV AUC = {search.best_score_:.4f}  {search.best_params_}")
    return search

spaces = {
    "Random forest": (RandomForestClassifier(n_estimators=300, random_state=SEED), False, 20, {
        "model__max_depth": [None, 14, 20, 25, 40], "model__min_samples_leaf": [1, 2, 4, 8],
        "model__max_features": ["sqrt", "log2", 0.3]}),
    "Extra trees": (ExtraTreesClassifier(n_estimators=300, random_state=SEED), False, 20, {
        "model__max_depth": [None, 14, 25, 40], "model__min_samples_leaf": [1, 2, 4, 8, 16],
        "model__max_features": ["sqrt", "log2", 0.3]}),
    "Gradient boosting": (HistGradientBoostingClassifier(random_state=SEED), False, 20, {
        "model__learning_rate": [0.03, 0.05, 0.1], "model__max_leaf_nodes": [15, 31, 63],
        "model__min_samples_leaf": [10, 20, 50], "model__l2_regularization": [0, 0.1, 1],
        "model__max_iter": [200, 400]}),
    "Logistic regression": (LogisticRegression(max_iter=2000), True, 5, {
        "model__C": [0.01, 0.1, 1, 10, 100]}),
}
searches = {n: tune(n, est, sp, scale, it) for n, (est, scale, it, sp) in spaces.items()}
tuned = {n: s.best_estimator_ for n, s in searches.items()}

## 4. Build ensembles and compare
* **Soft vote (RF + ET):** the original ensemble, averaging probabilities of two bagged-tree models.
* **Soft vote (all four):** adds boosting and a linear model, which make *different* kinds of errors.
* **Stacking:** a logistic regression learns how to weight the base models' out-of-fold probabilities.

An ensemble only helps if its members disagree in useful ways, so the comparison against single models is the evidence that the ensembling step is worth it.

In [ ]:
def fresh(name):  # unfitted copy of a tuned pipeline
    return clone(tuned[name])

base = list(tuned)
models = {n: fresh(n) for n in base}
models["Soft vote (RF+ET)"] = VotingClassifier(
    [("rf", fresh("Random forest")), ("et", fresh("Extra trees"))], voting="soft")
models["Soft vote (all four)"] = VotingClassifier([(n[:3], fresh(n)) for n in base], voting="soft")
models["Stacking (all four)"] = StackingClassifier(
    [(n[:3], fresh(n)) for n in base], final_estimator=LogisticRegression(max_iter=2000),
    cv=5, stack_method="predict_proba")

cv_scores = {n: cross_val_score(m, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
             for n, m in models.items()}
comparison = (pd.DataFrame({"mean_cv_auc": {n: s.mean() for n, s in cv_scores.items()},
                            "std": {n: s.std() for n, s in cv_scores.items()}})
              .sort_values("mean_cv_auc"))
display(comparison.round(4))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(comparison.index, comparison["mean_cv_auc"], xerr=comparison["std"], capsize=3, color="#4C78A8")
ax.set_xlim(comparison["mean_cv_auc"].min() - 0.03, min(1, comparison["mean_cv_auc"].max() + 0.02))
ax.set_xlabel("5-fold CV ROC AUC (error bars = fold-to-fold std; axis zoomed)")
ax.set_title("Single models vs. ensembles")
ax.grid(axis="x", alpha=.4); plt.tight_layout(); plt.show()

**How to read this:** gaps smaller than the fold-to-fold std are within noise. The final submission is chosen only from the ensemble rows (soft vote / stacking), even if a single model such as gradient boosting scores slightly higher, because the task requires ensemble learning. If a single model is clearly better than every ensemble, say so in the report and note that the ensemble includes it as a member.

In [ ]:
# Only ensembles of multiple models are eligible for the final submission (task requires ensemble learning)
ensemble_names = [n for n in comparison.index if n.startswith(("Soft vote", "Stacking"))]
best_name = comparison.loc[ensemble_names, "mean_cv_auc"].idxmax()
print("Selected ensemble:", best_name)
print("Best single model for reference:", comparison.drop(ensemble_names)["mean_cv_auc"].idxmax())

## 5. Honest check on the held-out validation split

In [ ]:
y_val_bin = (y_val == 2).astype(int)
fitted, rows = {}, []
plt.figure(figsize=(7, 6))
for n, m in models.items():
    m = clone(m).fit(X_train, y_train); fitted[n] = m
    p = m.predict_proba(X_val)[:, list(m.classes_).index(2)]
    auc = roc_auc_score(y_val_bin, p); rows.append((n, auc))
    fpr, tpr, _ = roc_curve(y_val_bin, p)
    plt.plot(fpr, tpr, lw=2.5 if n == best_name else 1, label=f"{n} (AUC {auc:.4f})")
plt.plot([0, 1], [0, 1], "k--", label="Chance")
plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("Validation ROC curves"); plt.legend(fontsize=8, loc="lower right"); plt.grid(alpha=.4); plt.show()
print(pd.DataFrame(rows, columns=["model", "validation_auc"]).sort_values("validation_auc").to_string(index=False))

## 6. Which features matter? (permutation importance of the selected model)

In [ ]:
imp = permutation_importance(fitted[best_name], X_val, y_val, scoring="roc_auc",
                             n_repeats=5, random_state=SEED, n_jobs=-1)
order = np.argsort(imp.importances_mean)[-15:]
plt.figure(figsize=(7, 5))
plt.barh(X.columns[order], imp.importances_mean[order], xerr=imp.importances_std[order], color="#4C78A8")
plt.xlabel("Drop in validation AUC when the feature is shuffled")
plt.title(f"Top features: {best_name}"); plt.grid(axis="x", alpha=.4); plt.tight_layout(); plt.show()

## 7. Refit on all labeled data and write the submission file

In [ ]:
final_model = clone(models[best_name]).fit(X, y)   # use every labeled row now that tuning is done
idx = list(final_model.classes_).index(2)
test_prob = final_model.predict_proba(test)[:, idx]

assert len(test_prob) == len(test), "row count mismatch"
print(f"Predictions: {len(test_prob)} | min {test_prob.min():.4f} | max {test_prob.max():.4f}")

# One column of Class-2 probabilities, no header, same row order as the test file
pd.DataFrame({"Class2_probability": test_prob}).to_csv("test_result_HW6Ensemble.csv", index=False, header=False)